In [1]:
# 1. Import required libraries & Verification
import os
import sys
import pandas as pd
import numpy as np

print("Python Kernel is working!")
print("Python version:", sys.version)
print("Python location:", sys.executable)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Python Kernel is working!
Python version: 3.14.8 (tags/v3.14.8:8e6e75d, Sep 30 2026, 18:19:33) [MSC v.1944 64 bit (AMD64)]
Python location: C:\Python314\python.exe
Pandas version: 3.1.0
NumPy version: 2.5.4


In [2]:
# 2. Location of CSV files 

data_path = "data/"
print("Data folder:", os.path.abspath(data_path))
print("Folder exists:", os.path.exists(data_path))


Data folder: C:\PythonTraining\Logistics_Dashboard\data
Folder exists: True


In [3]:
# 3. CAPSTONE Project is Opntion B - Load All CSV datasets.

# 3.1 Load all seven CSV datasets

customers = pd.read_csv(data_path + "customers.csv")
products = pd.read_csv(data_path + "products.csv")
orders = pd.read_csv(data_path + "orders.csv")
order_items = pd.read_csv(data_path + "order_items.csv")
payments = pd.read_csv(data_path + "payments.csv")
shipments = pd.read_csv(data_path + "shipments.csv")
returns = pd.read_csv(data_path + "returns.csv")

print("All seven datasets loaded successfully!")

datasets = {
    "Customers": customers,
    "Products": products,
    "Orders": orders,
    "Order Items": order_items,
    "Payments": payments,
    "Shipments": shipments,
    "Returns": returns
}

# Preserve original shipment data
original_shipments = shipments.copy()

All seven datasets loaded successfully!


In [4]:
# 4. Inspect dataset sizes and column names

for name, df in datasets.items():
    print(f"\n{name}")
    print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
    print("Column names:", df.columns.tolist())



Customers
Rows: 5,050 | Columns: 13
Column names: ['customer_id', 'first_name', 'last_name', 'gender', 'age', 'email', 'phone', 'region_code', 'region_name', 'city', 'barangay', 'signup_date', 'loyalty_tier']

Products
Rows: 800 | Columns: 9
Column names: ['product_id', 'product_name', 'brand', 'category', 'unit_cost', 'unit_price', 'stock_qty', 'uom', 'launch_date']

Orders
Rows: 15,075 | Columns: 8
Column names: ['order_id', 'customer_id', 'order_datetime', 'order_status', 'sales_channel', 'shipping_method', 'discount_pct', 'shipping_fee']

Order Items
Rows: 37,763 | Columns: 5
Column names: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price_at_purchase']

Payments
Rows: 15,000 | Columns: 6
Column names: ['payment_id', 'order_id', 'payment_method', 'payment_status', 'payment_date', 'amount_paid']

Shipments
Rows: 15,000 | Columns: 7
Column names: ['shipment_id', 'order_id', 'courier', 'tracking_no', 'shipment_status', 'ship_date', 'delivered_date']

Returns
Rows: 1,

In [5]:
# 5. Missing values assessment

for name, df in datasets.items():
    print(f"\n========== {name} ==========")

    missing = df.isnull().sum()

    missing_summary = pd.DataFrame({
        "Missing Values": missing,
        "Missing Percentage": (missing / len(df) * 100).round(2)
    })

    print(missing_summary[missing_summary["Missing Values"] > 0])



========== Customers ==========
              Missing Values  Missing Percentage
gender                   301                5.96
age                      150                2.97
email                    107                2.12
phone                    147                2.91
loyalty_tier             266                5.27

========== Products ==========
           Missing Values  Missing Percentage
stock_qty              19                2.38

========== Orders ==========
              Missing Values  Missing Percentage
discount_pct             758                5.03
shipping_fee             474                3.14

========== Order Items ==========
                        Missing Values  Missing Percentage
unit_price_at_purchase            2262                5.99

========== Payments ==========
             Missing Values  Missing Percentage
amount_paid             878                5.85

========== Shipments ==========
                Missing Values  Missing Percentage
deliver

In [6]:

# 6. Check duplicate records and primary IDs

primary_keys = {
    "Customers": "customer_id",
    "Products": "product_id",
    "Orders": "order_id",
    "Order Items": "order_item_id",
    "Payments": "payment_id",
    "Shipments": "shipment_id",
    "Returns": "return_id"
}

for name, df in datasets.items():
    key = primary_keys[name]

    print(f"\n{name}")
    print("Duplicate rows:", df.duplicated().sum())
    print(f"Duplicate {key}:", df[key].duplicated().sum())


Customers
Duplicate rows: 50
Duplicate customer_id: 50

Products
Duplicate rows: 0
Duplicate product_id: 0

Orders
Duplicate rows: 75
Duplicate order_id: 75

Order Items
Duplicate rows: 262
Duplicate order_item_id: 262

Payments
Duplicate rows: 0
Duplicate payment_id: 0

Shipments
Duplicate rows: 0
Duplicate shipment_id: 0

Returns
Duplicate rows: 0
Duplicate return_id: 0


In [7]:

# 7. Identify orphan records

# Check Orders with missing customer references
orphan_customers = orders[
    ~orders["customer_id"].isin(customers["customer_id"])
]

# Check Order Items with missing product references
orphan_products = order_items[
    ~order_items["product_id"].isin(products["product_id"])
]

print("Orphan customer records:", len(orphan_customers))
print("Orphan product records:", len(orphan_products))

Orphan customer records: 43
Orphan product records: 129


In [8]:
# 8. Create working copies and remove exact duplicates

cleaned_datasets = {}

for name, df in datasets.items():

    before = len(df)

    # Remove exact duplicate rows
    cleaned_df = df.drop_duplicates().copy()

    after = len(cleaned_df)

    cleaned_datasets[name] = cleaned_df

    print(f"{name}:")
    print(f"  Original records: {before}")
    print(f"  Cleaned records: {after}")
    print(f"  Removed duplicates: {before - after}")
    print()

Customers:
  Original records: 5050
  Cleaned records: 5000
  Removed duplicates: 50

Products:
  Original records: 800
  Cleaned records: 800
  Removed duplicates: 0

Orders:
  Original records: 15075
  Cleaned records: 15000
  Removed duplicates: 75

Order Items:
  Original records: 37763
  Cleaned records: 37501
  Removed duplicates: 262

Payments:
  Original records: 15000
  Cleaned records: 15000
  Removed duplicates: 0

Shipments:
  Original records: 15000
  Cleaned records: 15000
  Removed duplicates: 0

Returns:
  Original records: 1200
  Cleaned records: 1200
  Removed duplicates: 0



In [9]:
# 9. Initialize working DataFrames

payments_df = cleaned_datasets["Payments"]
shipments_df = cleaned_datasets["Shipments"]
returns_df = cleaned_datasets["Returns"]

In [32]:
# 10. Fill missing customer information
customer_df = cleaned_datasets["Customers"]

customer_df["email"] = customer_df["email"].fillna("default@gmail.com")
customer_df["phone"] = customer_df["phone"].fillna("09100000001")

customer_df["gender"] = customer_df["gender"].fillna("Unknown")
customer_df["loyalty_tier"] = customer_df["loyalty_tier"].fillna("Unknown")

customer_df["phone"] = customer_df["phone"].astype("string")

print("Customer missing values updated successfully!")
print("Phone data type:", customer_df["phone"].dtype)

# Verify the results
print("\nRemaining missing values:")
print(customer_df[["email", "phone", "gender", "loyalty_tier"]].isna().sum())

Customer missing values updated successfully!
Phone data type: string

Remaining missing values:
email           0
phone           0
gender          0
loyalty_tier    0
dtype: int64


In [11]:
# 11. Inspect missing product stock quantities

product_df = cleaned_datasets["Products"]

print("Missing stock quantities:",
      product_df["stock_qty"].isna().sum())

print("\nStock quantity summary:")
print(product_df["stock_qty"].describe())

Missing stock quantities: 19

Stock quantity summary:
count    781.000000
mean     121.014085
std       59.075645
min        0.000000
25%       81.000000
50%      118.000000
75%      162.000000
max      322.000000
Name: stock_qty, dtype: float64


In [12]:
# 12. Inspect product category values

print("Product categories:")
print(product_df["category"].value_counts(dropna=False))

print("\nUnique categories:")
print(product_df["category"].unique())

Product categories:
category
Beauty              94
Toys                92
Books               90
Fashion             89
Groceries           77
Home & Kitchen      76
Sports              74
Electronics         71
electronics         20
Book                20
Home and Kitchen    19
Electronics         17
Grocery             12
Elec                11
Beauty & Care       11
Toy                 11
Fashon               8
Sports & Outdoor     8
Name: count, dtype: int64

Unique categories:
<ArrowStringArray>
[          'Beauty', 'Home and Kitchen',          'Fashion',
             'Toys',            'Books',           'Fashon',
        'Groceries',      'electronics',           'Sports',
      'Electronics',          'Grocery',     'Electronics ',
   'Home & Kitchen',             'Elec',    'Beauty & Care',
              'Toy',             'Book', 'Sports & Outdoor']
Length: 18, dtype: str


In [13]:
# 13. Standardize product categories

# Preserve original category values
product_df["category_original"] = product_df["category"].copy()

# Remove extra spaces and standardize capitalization
product_df["category"] = (
    product_df["category"]
    .str.strip()
    .str.lower()
)

# Map inconsistent values to standard categories
category_mapping = {
    "electronics": "Electronics",
    "elec": "Electronics",
    "books": "Books",
    "book": "Books",
    "toys": "Toys",
    "toy": "Toys",
    "fashion": "Fashion",
    "fashon": "Fashion",
    "groceries": "Groceries",
    "grocery": "Groceries",
    "home & kitchen": "Home & Kitchen",
    "home and kitchen": "Home & Kitchen",
    "beauty": "Beauty",
    "beauty & care": "Beauty",
    "sports": "Sports",
    "sports & outdoor": "Sports"
}
product_df["category"] = product_df["category"].replace(category_mapping)

print("Product categories standardized successfully!")

print("\nStandardized categories:")
print(product_df["category"].value_counts())

Product categories standardized successfully!

Standardized categories:
category
Electronics       119
Books             110
Beauty            105
Toys              103
Fashion            97
Home & Kitchen     95
Groceries          89
Sports             82
Name: count, dtype: int64


In [14]:
# 14. Inspect Order Items unit prices

order_items_df = cleaned_datasets["Order Items"]

print("Missing unit prices:",
      order_items_df["unit_price_at_purchase"].isna().sum())

print("\nUnit price summary:")
print(order_items_df["unit_price_at_purchase"].describe())

print("\nSample unit price values:")
print(order_items_df["unit_price_at_purchase"].dropna().head(20))

print("\nData type:")
print(order_items_df["unit_price_at_purchase"].dtype)

print("\nSample unique values:")
print(order_items_df["unit_price_at_purchase"].dropna().unique()[:20])

Missing unit prices: 2236

Unit price summary:
count      35265
unique       781
top       100.62
freq         105
Name: unit_price_at_purchase, dtype: object

Sample unit price values:
0     120.67
1      83.58
2      46.96
3     100.59
4     109.18
5      54.41
6      85.94
7     117.13
9      69.87
10     73.02
11     30.07
12     90.42
13    158.03
14    ₱69.98
15     71.01
16     30.07
17     31.49
18     84.81
19     39.99
20    106.11
Name: unit_price_at_purchase, dtype: str

Data type:
str

Sample unique values:
<ArrowStringArray>
['120.67',  '83.58',  '46.96', '100.59', '109.18',  '54.41',  '85.94',
 '117.13',  '69.87',  '73.02',  '30.07',  '90.42', '158.03', '₱69.98',
  '71.01',  '31.49',  '84.81',  '39.99', '106.11',  '42.05']
Length: 20, dtype: str


In [15]:
# 15. Clean and convert unit prices

# Rebuild from original CSV and remove exact duplicates
order_items_df = pd.read_csv(
    data_path + "order_items.csv"
).drop_duplicates().copy()

# Remove Peso prefix and convert to numeric
order_items_df["unit_price_at_purchase"] = pd.to_numeric(
    order_items_df["unit_price_at_purchase"]
    .astype("string")
    .str.replace(r"^[P₱]\s*", "", regex=True),
    errors="coerce"
)

# Update cleaned dataset
cleaned_datasets["Order Items"] = order_items_df

# Verify results
print("Unit price cleaning completed!")
print("Total records:", len(order_items_df))
print("Data type:", order_items_df["unit_price_at_purchase"].dtype)
print("Missing values:", order_items_df["unit_price_at_purchase"].isna().sum())

Unit price cleaning completed!
Total records: 37501
Data type: Float64
Missing values: 2236


In [16]:
# 16. Inspect, clean and verify payment amounts

payments_df = cleaned_datasets["Payments"].copy()

# Part 1: Inspect
print("Missing payment amounts:",
      payments_df["amount_paid"].isna().sum())

print("\nSample payment amounts:")
print(payments_df["amount_paid"].dropna().head(10))

# Part 2: Clean
payments_df["amount_paid"] = (
    payments_df["amount_paid"]
    .astype("string")
    .str.replace("₱", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

payments_df["amount_paid"] = pd.to_numeric(
    payments_df["amount_paid"],
    errors="coerce"
)

# Part 3: Verify payment amounts

print("Sample payment amounts:")
print(payments_df["amount_paid"].head(10))

print("\nData type:")
print(payments_df["amount_paid"].dtype)

print("\nMissing payment amounts:")
print(payments_df["amount_paid"].isna().sum())

# Save cleaned payment data
cleaned_datasets["Payments"] = payments_df

Missing payment amounts: 878

Sample payment amounts:
0      2822.72
1      2499.76
2      1983.66
3      1765.99
4      1477.53
5       1422.1
6      2003.03
7    ₱2,250.92
8       288.25
9      2372.75
Name: amount_paid, dtype: str
Sample payment amounts:
0    2822.72
1    2499.76
2    1983.66
3    1765.99
4    1477.53
5     1422.1
6    2003.03
7    2250.92
8     288.25
9    2372.75
Name: amount_paid, dtype: Float64

Data type:
Float64

Missing payment amounts:
878


In [17]:
# 17. Inspect missing shipment delivery dates

# Count missing delivered dates
missing_dates = shipments_df["delivered_date"].isna().sum()

print("Missing delivered dates:", missing_dates)

# Check shipment status of records with missing dates
missing_date_records = shipments_df[
    shipments_df["delivered_date"].isna()
]

print("\nShipment status of records with missing dates:")
print(missing_date_records["shipment_status"].value_counts())

# Display sample records
print("\nSample records with missing delivered dates:")
display(missing_date_records.head(10))

Missing delivered dates: 729

Shipment status of records with missing dates:
shipment_status
Delivered             558
In Transit            137
Returned to Sender     24
Lost                   10
Name: count, dtype: int64

Sample records with missing delivered dates:


,shipment_id,order_id,courier,tracking_no,shipment_status,ship_date,delivered_date
24,SHP0000025,ORD0000025,GrabExpress,04517LZLQF7U,Delivered,2024-03-19,NaN
27,SHP0000028,ORD0000028,J&T,CZKK425MUQYH,Delivered,2024-03-31,NaN
38,SHP0000039,ORD0000039,J&T,CTU7TEG2SA4R,Delivered,13-Oct-2024,NaN
64,SHP0000065,ORD0000065,NinjaVan,32AGND4SDBW2,Delivered,2024-01-19,NaN
71,SHP0000072,ORD0000072,LBC,Q2WD7BACWAMZ,Delivered,2025-04-03,NaN
87,SHP0000088,ORD0000088,GrabExpress,4NALWDMUFKHH,Delivered,2024-06-19,NaN
91,SHP0000092,ORD0000092,NinjaVan,0R9B9OTGE44A,Delivered,2026-01-15,NaN
94,SHP0000095,ORD0000095,J&T,GDMK9TBGJPWV,In Transit,2025-12-16,NaN
136,SHP0000137,ORD0000137,NinjaVan,7AWHNMR1C433,Delivered,2025-01-29,NaN
179,SHP0000180,ORD0000180,LBC,BG3UMVTGLQJG,In Transit,2024-06-27,NaN


In [18]:
# 18 Correct shipment date formats

# Reload original data and remove duplicates

shipments_df = original_shipments.drop_duplicates().copy()

# Function to handle the three date formats
def clean_shipment_dates(series):

    dates = series.astype("string").str.strip()

    # Format 1: YYYY-MM-DD
    result = pd.to_datetime(
        dates, format="%Y-%m-%d", errors="coerce"
    )

    # Format 2: DD-MMM-YYYY
    missing = result.isna() & dates.notna()
    result.loc[missing] = pd.to_datetime(
        dates.loc[missing],
        format="%d-%b-%Y",
        errors="coerce"
    )

    # Format 3: MM/DD/YYYY
    missing = result.isna() & dates.notna()
    result.loc[missing] = pd.to_datetime(
        dates.loc[missing],
        format="%m/%d/%Y",
        errors="coerce"
    )

    return result


# Apply date cleaning
shipments_df["ship_date"] = clean_shipment_dates(
    shipments_df["ship_date"]
)

shipments_df["delivered_date"] = clean_shipment_dates(
    shipments_df["delivered_date"]
)

# Update cleaned dataset
cleaned_datasets["Shipments"] = shipments_df

# Verify results
print("Shipment date conversion completed!")

print("\nDate data types:")
print(shipments_df[["ship_date", "delivered_date"]].dtypes)

print("\nMissing dates:")
print(shipments_df[["ship_date", "delivered_date"]].isna().sum())

Shipment date conversion completed!

Date data types:
ship_date         datetime64[us]
delivered_date    datetime64[us]
dtype: object

Missing dates:
ship_date           0
delivered_date    729
dtype: int64


In [19]:


# 19. Check shipment status and delivery date consistency

status_date_check = shipments_df.groupby(
    "shipment_status"
)["delivered_date"].agg(
    Total_Records="count"
)

print("Delivery dates by shipment status:")
print(status_date_check)

print("\nNon-Delivered shipments with delivery dates:")

non_delivered = shipments_df[
    (shipments_df["shipment_status"] != "Delivered") &
    (shipments_df["delivered_date"].notna())
]

print(non_delivered["shipment_status"].value_counts())
print("Total affected records:", len(non_delivered))

Delivery dates by shipment status:
                    Total_Records
shipment_status                  
Delivered                   11122
In Transit                   2557
Lost                          151
Returned to Sender            441

Non-Delivered shipments with delivery dates:
shipment_status
In Transit            2557
Returned to Sender     441
Lost                   151
Name: count, dtype: int64
Total affected records: 3149


In [20]:

# 20 Validate delivery date sequence

shipments_df["delivery_lead_time"] = (
    shipments_df["delivered_date"] -
    shipments_df["ship_date"]
).dt.total_seconds() / 86400

negative_lead_time = shipments_df[
    shipments_df["delivery_lead_time"] < 0
]

print("Negative lead time records:",
      len(negative_lead_time))

print("\nNegative lead time by shipment status:")
print(negative_lead_time["shipment_status"].value_counts())

print("\nSample negative records:")
print(negative_lead_time[
    ["shipment_id", "shipment_status",
     "ship_date", "delivered_date", "delivery_lead_time"]
].head(10))

Negative lead time records: 0

Negative lead time by shipment status:
Series([], Name: count, dtype: int64)

Sample negative records:
Empty DataFrame
Columns: [shipment_id, shipment_status, ship_date, delivered_date, delivery_lead_time]
Index: []


In [21]:

# 21. Calculate valid delivery lead time

shipments_df["valid_delivery_lead_time"] = np.where(
    (shipments_df["shipment_status"] == "Delivered") &
    (shipments_df["delivered_date"].notna()) &
    (shipments_df["ship_date"].notna()),
    shipments_df["delivery_lead_time"],
    np.nan
)

# Update cleaned dataset
cleaned_datasets["Shipments"] = shipments_df

# Verify results
print("Valid delivery records:",
      shipments_df["valid_delivery_lead_time"].notna().sum())

print("\nValid delivery lead time summary:")
print(shipments_df["valid_delivery_lead_time"].describe())

Valid delivery records: 11122

Valid delivery lead time summary:
count    11122.000000
mean         4.003507
std          1.993129
min          1.000000
25%          2.000000
50%          4.000000
75%          6.000000
max          7.000000
Name: valid_delivery_lead_time, dtype: float64


In [22]:

# 22. Inspect Returns data

# Part 1: Check missing values
print("Missing refunded_flag:",
      returns_df["refunded_flag"].isna().sum())

print("Missing refund_amount:",
      returns_df["refund_amount"].isna().sum())

# Part 2: Check refund status counts
print("\nRefund status counts:")
print(returns_df["refunded_flag"].value_counts(dropna=False))

# Part 3: Inspect refund amounts by status
print("\nRefund amount summary by status:")
print(
    returns_df.groupby("refunded_flag")["refund_amount"]
    .describe()
)

# Part 4: Display sample records
print("\nSample Returns records:")
display(returns_df.head(10))

Missing refunded_flag: 110
Missing refund_amount: 187

Refund status counts:
refunded_flag
Yes        653
No         301
Partial    136
NaN        110
Name: count, dtype: int64

Refund amount summary by status:
               count        mean         std  min      25%      50%       75%  \
refunded_flag                                                                   
No             248.0  638.657742  410.091637  0.0  319.700  615.325  928.6650   
Partial        119.0  642.806050  373.264118  0.0  389.055  660.050  890.7550   
Yes            554.0  688.314621  426.141332  0.0  365.145  679.390  960.2725   

                   max  
refunded_flag           
No             1809.38  
Partial        1464.86  
Yes            2194.43  

Sample Returns records:


,return_id,order_id,return_request_date,reason,refunded_flag,refund_amount
0,RET000001,ORD0008752,2025-01-09,Defective,No,NaN
1,RET000002,ORD0007359,2025-07-20,Wrong item,NaN,650.28
2,RET000003,ORD0009041,01/07/2025,Size issue,Yes,NaN
3,RET000004,ORD0000227,2025-03-04,Late delivery,Partial,NaN
4,RET000005,ORD0005754,2025-04-24,Size issue,Yes,713.39
5,RET000006,ORD0002714,2024-07-05,Size issue,Partial,675.23
6,RET000007,ORD0009572,2025-02-07,Missing parts,No,931.12
7,RET000008,ORD0005308,2024-12-23,Defective,Yes,209.83
8,RET000009,ORD0007574,2025-08-08,Wrong item,Yes,NaN
9,RET000010,ORD0003302,2024-06-05,Changed mind,No,1048.55


In [23]:

# 23. Fill missing refund status

returns_df["refunded_flag"] = (
    returns_df["refunded_flag"].fillna("Unknown")
)

# Update cleaned dataset
cleaned_datasets["Returns"] = returns_df

# Verify results
print("Refund status cleaning completed!")

print("\nUpdated refund status distribution:")
print(returns_df["refunded_flag"].value_counts())

print("\nRemaining missing refund flags:",
      returns_df["refunded_flag"].isna().sum())

Refund status cleaning completed!

Updated refund status distribution:
refunded_flag
Yes        653
No         301
Partial    136
Unknown    110
Name: count, dtype: int64

Remaining missing refund flags: 0


In [24]:

# 24. Review refund amounts by refund status

print("Refund amount summary by status:")

print(
    returns_df.groupby("refunded_flag")["refund_amount"]
    .agg(["count", "mean", "min", "max"])
)

print("\nMissing refund amounts by status:")

print(
    returns_df.groupby("refunded_flag")["refund_amount"]
    .apply(lambda x: x.isna().sum())
)

Refund amount summary by status:
               count        mean  min      max
refunded_flag                                 
No               248  638.657742  0.0  1809.38
Partial          119  642.806050  0.0  1464.86
Unknown           92  630.904348  0.0  1797.65
Yes              554  688.314621  0.0  2194.43

Missing refund amounts by status:
refunded_flag
No         53
Partial    17
Unknown    18
Yes        99
Name: refund_amount, dtype: int64


In [25]:

# 25. Final primary-key and foreign-key validation

# Part 1: Primary key validation

primary_keys = {
    "Customers": "customer_id",
    "Products": "product_id",
    "Orders": "order_id",
    "Order Items": "order_item_id",
    "Payments": "payment_id",
    "Shipments": "shipment_id",
    "Returns": "return_id"
}

print("PRIMARY KEY VALIDATION")
print("=" * 45)

for name, key in primary_keys.items():

    df = cleaned_datasets[name]

    duplicate_ids = df[key].duplicated().sum()
    missing_ids = df[key].isna().sum()

    print(f"\n{name}")
    print("Duplicate IDs:", duplicate_ids)
    print("Missing IDs:", missing_ids)


# Part 2: Foreign key validation

relationships = [
    ("Orders", "customer_id", "Customers", "customer_id"),
    ("Order Items", "order_id", "Orders", "order_id"),
    ("Order Items", "product_id", "Products", "product_id"),
    ("Payments", "order_id", "Orders", "order_id"),
    ("Shipments", "order_id", "Orders", "order_id"),
    ("Returns", "order_id", "Orders", "order_id")
]

print("\n\nFOREIGN KEY VALIDATION")
print("=" * 50)

for child, child_key, parent, parent_key in relationships:

    child_df = cleaned_datasets[child]
    parent_df = cleaned_datasets[parent]

    missing_keys = child_df[child_key].isna().sum()

    orphan_count = (
        child_df[child_key].notna()
        & ~child_df[child_key].isin(parent_df[parent_key].dropna())
    ).sum()

    print(f"\n{child} -> {parent}")
    print("Missing keys:", missing_keys)
    print("Orphan records:", orphan_count)

PRIMARY KEY VALIDATION

Customers
Duplicate IDs: 0
Missing IDs: 0

Products
Duplicate IDs: 0
Missing IDs: 0

Orders
Duplicate IDs: 0
Missing IDs: 0

Order Items
Duplicate IDs: 0
Missing IDs: 0

Payments
Duplicate IDs: 0
Missing IDs: 0

Shipments
Duplicate IDs: 0
Missing IDs: 0

Returns
Duplicate IDs: 0
Missing IDs: 0


FOREIGN KEY VALIDATION

Orders -> Customers
Missing keys: 0
Orphan records: 43

Order Items -> Orders
Missing keys: 0
Orphan records: 0

Order Items -> Products
Missing keys: 0
Orphan records: 129

Payments -> Orders
Missing keys: 0
Orphan records: 0

Shipments -> Orders
Missing keys: 0
Orphan records: 0

Returns -> Orders
Missing keys: 0
Orphan records: 0


In [26]:

# 26. Validate data types across all datasets

print("DATA TYPE VALIDATION")
print("=" * 50)

for name, df in cleaned_datasets.items():

    print(f"\n========== {name} ==========")

    for column in df.columns:
        print(f"{column}: {df[column].dtype}")

DATA TYPE VALIDATION

========== Customers ==========
customer_id: str
first_name: str
last_name: str
gender: str
age: float64
email: str
phone: string
region_code: str
region_name: str
city: str
barangay: str
signup_date: str
loyalty_tier: str

========== Products ==========
product_id: str
product_name: str
brand: str
category: str
unit_cost: float64
unit_price: str
stock_qty: float64
uom: str
launch_date: str
category_original: str

========== Orders ==========
order_id: str
customer_id: str
order_datetime: str
order_status: str
sales_channel: str
shipping_method: str
discount_pct: float64
shipping_fee: float64

========== Order Items ==========
order_item_id: str
order_id: str
product_id: str
quantity: int64
unit_price_at_purchase: Float64

========== Payments ==========
payment_id: str
order_id: str
payment_method: str
payment_status: str
payment_date: str
amount_paid: Float64

========== Shipments ==========
shipment_id: str
order_id: str
courier: str
tracking_no: str
shipment_st

In [27]:

# 27. Check selected columns requiring conversion

date_columns = {
    "Customers": ["signup_date"],
    "Products": ["launch_date"],
    "Orders": ["order_datetime"],
    "Payments": ["payment_date"],
     "Returns": ["return_request_date"]
}

for name, columns in date_columns.items():

    df = cleaned_datasets[name]

    print(f"\n{name}")

    for column in columns:
        print(column, ":", df[column].dtype)


Customers
signup_date : str

Products
launch_date : str

Orders
order_datetime : str

Payments
payment_date : str

Returns
return_request_date : str


In [28]:

# 28. Inspect date formats and preview conversion

for name, columns in date_columns.items():

    df = cleaned_datasets[name]

    for column in columns:

        print(f"\n{name} - {column}")
        print("Original data type:", df[column].dtype)

        print("\nSample date values:")
        print(df[column].dropna().head(5).tolist())

        # Preview date conversion
        converted_dates = pd.to_datetime(
            df[column],
            format="mixed",
            dayfirst=False,
            errors="coerce"
        )

        print("\nInvalid date values:",
              converted_dates.isna().sum() - df[column].isna().sum())

        print("New missing values:",
              converted_dates.isna().sum())


Customers - signup_date
Original data type: str

Sample date values:
['2025-12-31', '2025-02-08', '2023-09-10', '2024-11-17', '2023-01-21']

Invalid date values: 0
New missing values: 0

Products - launch_date
Original data type: str

Sample date values:
['2022-01-18', '2022-11-25', '2023-09-04', '2023-08-12', '2024-05-12']

Invalid date values: 0
New missing values: 0

Orders - order_datetime
Original data type: str

Sample date values:
['2025-11-07 17:02:00', '09/18/2025 09:13 AM', '2025-08-08 04:59:00', '2025-10-14 12:56:00', '2024-05-30 04:56:00']

Invalid date values: 0
New missing values: 0

Payments - payment_date
Original data type: str

Sample date values:
['2024-05-09', '06/25/2024', '2025-06-10', '2025-06-21', '06/28/2024']

Invalid date values: 0
New missing values: 0

Returns - return_request_date
Original data type: str

Sample date values:
['2025-01-09', '2025-07-20', '01/07/2025', '2025-03-04', '2025-04-24']

Invalid date values: 0
New missing values: 0


In [29]:

# 29. Apply date conversions

for name, columns in date_columns.items():

    df = cleaned_datasets[name]

    for column in columns:

        df[column] = pd.to_datetime(
            df[column],
            format="mixed",
            dayfirst=False,
            errors="coerce"
        )

    cleaned_datasets[name] = df

print("Date conversion completed successfully!")

Date conversion completed successfully!


In [30]:

# 30. Verify final date data types

for name, columns in date_columns.items():

    df = cleaned_datasets[name]

    print(f"\n{name}")

    for column in columns:
        print(
            column,
            ":",
            df[column].dtype,
            "| Missing:",
            df[column].isna().sum()
        )


Customers
signup_date : datetime64[us] | Missing: 0

Products
launch_date : datetime64[us] | Missing: 0

Orders
order_datetime : datetime64[us] | Missing: 0

Payments
payment_date : datetime64[us] | Missing: 0

Returns
return_request_date : datetime64[us] | Missing: 0


In [31]:
# 31. Save cleaned datasets

import os

# Create output folder
output_path = "cleaned_data"
os.makedirs(output_path, exist_ok=True)

# Save each cleaned dataset as CSV
for name, df in cleaned_datasets.items():

    file_name = name.lower().replace(" ", "_") + ".csv"

    df.to_csv(
        os.path.join(output_path, file_name),
        index=False
    )

    print(f"Saved: {file_name}")

print("\nAll cleaned datasets saved successfully!")

Saved: customers.csv
Saved: products.csv
Saved: orders.csv
Saved: order_items.csv
Saved: payments.csv
Saved: shipments.csv
Saved: returns.csv

All cleaned datasets saved successfully!
